# C1 NER replication (WikiANN-en)
Does the SentencePiece QA-vs-BIO trailing-punctuation artifact replicate on NER?

Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom.

In [ ]:
# 1. Setup: Drive + repo + deps
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r Requirements.txt

In [ ]:
# 2. Preflight: outputs must land on Drive, data must be the detokenized build
import os, json
DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor_NER'
os.makedirs(DRIVE_OUT, exist_ok=True)
assert 'drive' in os.path.realpath(DRIVE_OUT).lower()
P = set(",.;:!?'\")]")
rows = [json.loads(l) for l in open('data/ner_wikiann_en/Splits/test.jsonl')]
rate = sum(r['span_end'] < len(r['sentence']) and r['sentence'][r['span_end']] in P for r in rows) / len(rows)
print(f'{len(rows)} test rows, {rate:.1%} gold spans followed by punctuation')
assert rate > 0.10, 'old space-joined data: artifact cannot occur, stop'

In [ ]:
# 3. Dry run (1 epoch, XLM-R). Check exact match is not ~0 before continuing.
# O_WEIGHT 0.615 keeps the O:span loss balance of the idiom runs (0.104); NER spans are 62% of words vs 21% for idioms
ENV = f"DRIVE_OUT={DRIVE_OUT} DATA_DIR=data/ner_wikiann_en/Splits LANGS=English TEST_LANGS=English O_WEIGHT=0.615"
!{ENV} OUT_SUB=flip MODEL=xlm-roberta-base ENC_SHORT=xlmr bash experiments/rigor/run_07_mdeberta_sp_replication.sh --dry-run
# If exact match ~0 (underfitting at the RemBERT LRs), uncomment, rerun this cell, and keep it for ALL encoders:
# ENV += ' LR_JOINT=3e-5 LR_BIO=3e-5'

In [ ]:
# 4. Clean dry-run dirs, then full seed-42 matrix (8 runs). Skips runs whose metrics.json already exists.
!rm -rf {DRIVE_OUT}/flip/_dryrun_joint {DRIVE_OUT}/flip/_dryrun_bio
SEEDS = '42'   # later: '123 7'
ENCODERS = [('google/rembert','rembert','rembert'), ('xlm-roberta-base','xlmr','flip'),
            ('bert-base-multilingual-cased','mbert','flip'), ('google/muril-base-cased','muril','flip')]
for model, short, sub in ENCODERS:
    !{ENV} SEEDS="{SEEDS}" OUT_SUB={sub} MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 5. Read back from Drive: every run must have fresh, non-empty predictions
import glob, time
for f in sorted(glob.glob(f'{DRIVE_OUT}/*/*_s*/test_predictions.jsonl')):
    n = sum(1 for _ in open(f))
    print(f'{n:4d} rows  {time.ctime(os.path.getmtime(f))}  {f}')

In [ ]:
# 6. Score: original / extend / strip gap per tokenizer family
!python experiments/rigor/run_08_extended_gold.py --preds-root {DRIVE_OUT} --seeds 42